# Lab 6: Combining Data

**DSA 405 · Week 6**

| | |
|---|---|
| **In class** | Friday, Sep 25 |
| **Take-home** | none; **P2 is due Thursday, Oct 1** |
| **Files** | `inspections_a.csv`, `inspections_b.csv` |
| **Also this week** | **Bench Check 1** window continues (through Week 7) |

> **No A this week.** Submit this notebook for in-class credit and use the week
> to finish **P2**: audit, data dictionary, cleaning log, Provenance Brief. The material
> below prepares you for P3, which requires joining two sources.

## Overview

Today we learn about the **fan-out join**, one of the most common silent data errors
and a frequent problem in AI-generated code. A "silent" error is one that produces no
error message. In a fan-out join, two tables are merged, the merge creates extra copies
of rows without any warning, and every statistic computed afterward is based on rows
that should not exist, while all the output still looks reasonable.

We're going to produce that error on purpose, measure exactly what it does, and then
add the two lines of code that will prevent it from happening.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

---
# Part 1: Explore (in class)

## Task 1.1: Know the grain before joining

The first question to ask about any table is: what does one row represent? The answer
is called the table's *grain*. Joining two tables before you can answer that question
for both of them is how the fan-out error happens, so we start by answering it:

In [ ]:
insp = load("inspections_a.csv")   # the inspections table
rest = load("inspections_b.csv")   # the restaurant table

print("inspections_a:", insp.shape)                       # .shape is (rows, columns)
# .is_unique is True when a column never repeats a value
print("  unique inspection_id :", insp.inspection_id.is_unique)
# .duplicated([...]) marks rows repeating an earlier restaurant-and-date pair,
# and .any() asks whether there are any such rows at all
print("  one row per restaurant per day:",
      not insp.duplicated(["restaurant_id", "inspection_date"]).any())
print()
print("inspections_b:", rest.shape)
print("  unique restaurant_id :", rest.restaurant_id.is_unique)
# .nunique() counts how many different values the column has
print("  distinct restaurants :", rest.restaurant_id.nunique())

Read the output carefully. Table A has 1,200 rows, one per inspection. Table B has
**1,452 rows but only 300 restaurants**, because its grain is one row per restaurant
*per cuisine tag*. A restaurant tagged `southern` and `seafood` appears twice:

In [ ]:
# .duplicated(keep=False) marks every row whose restaurant_id appears more than once,
# including the first one, so all copies of a restaurant stay together.
# .sort_values puts rows with the same restaurant_id next to each other.
example = rest[rest.restaurant_id.duplicated(keep=False)].sort_values("restaurant_id")
example.head(6)   # show the first 6 rows

## Task 1.2: The naive join

Every inspection needs its restaurant's name and city. A *naive* join is one written
without checking the grain of each table first, and it seems like the obvious approach:

In [ ]:
# how="left" keeps every row of insp and attaches the matching rows from rest
merged = insp.merge(rest, on="restaurant_id", how="left")

print(f"rows in : {len(insp):>6}")     # len() counts the rows in a table
print(f"rows out: {len(merged):>6}")

**1,200 rows in, 5,800 rows out.** Each inspection was duplicated once per cuisine tag,
4.83 copies on average, with no warning and no error message. This happens because a
left join is *defined* to work this way when the right-hand table has more than one
row for the same key.

The next cells show why this error is so hard to notice after it happens:

In [ ]:
# .mean() averages a column, and round(..., 2) keeps two decimal places
print("mean score, before:", round(insp.score.mean(), 2))
print("mean score, after :", round(merged.score.mean(), 2))
print()
# .groupby("city") sorts the rows into one group per city, .score.mean() averages each
# group, and .idxmin() returns the name of the group with the smallest average
print("worst city by mean score:", merged.groupby("city").score.mean().idxmin())
merged.head(3)

The mean barely changed, `.head()` looks normal, and every number a reader would
quickly check looks fine. But the merged data is wrong in three ways: restaurants with
several cuisine tags now count more than once in every group statistic, every count is
about 5× too large, and any claim about individual inspections is wrong. Nothing in the
notebook's output tells you that any of this happened, which is why we add two
protective lines of code next.

## Task 1.3: The two lines that prevent it

First line: write down the relationship you expect between the two tables, and have
pandas check it during the join.

In [ ]:
# try/except runs the code and catches the error, so the notebook keeps going
try:
    # validate= tells pandas to check the relationship before it merges anything
    insp.merge(rest, on="restaurant_id", how="left", validate="many_to_one")
except pd.errors.MergeError as e:   # pandas raises MergeError when the check fails
    print("MergeError:", e)

`validate="many_to_one"` states the relationship you expect: many inspections may share
one restaurant, but each restaurant may appear only once in the right-hand table. Here
it appears more than once, so the merge raises an error instead of fanning out. An
error here is the good outcome. Without it, you get 5,800 rows and no sign that
anything went wrong.

Second line: fix the grain, then check the result with an `assert`, a line of code
that states what must be true and raises an error if it is not.

In [ ]:
# .drop_duplicates("restaurant_id") keeps the first row for each restaurant and removes
# the others, so restaurant_id becomes unique. This keeps only the first cuisine tag.
# If you need all the tags, collect them into a list instead (you will decide in P3).
rest_one = rest.drop_duplicates("restaurant_id")

# the relationship is now genuinely many-to-one, so this merge passes the check
clean = insp.merge(rest_one, on="restaurant_id", how="left", validate="many_to_one")

# assert stops the notebook with an error if the statement after it is not true
assert len(clean) == len(insp), f"expected {len(insp)} rows, got {len(clean)}"
print(f"rows in: {len(insp)}, rows out: {len(clean)}. The assert will check this on every run.")

### The four things `validate=` can check

`validate=` takes one of four values. Each value names the relationship you expect
between the two tables, written as *left table* `_to_` *right table*. "One" means the key
appears at most once in that table. "Many" means it is allowed to appear more than once.

| Value | What you are promising | An example |
|---|---|---|
| `"one_to_one"` | The key appears at most once in **both** tables | One row per restaurant, joined to one permit per restaurant |
| `"one_to_many"` | The key is unique on the **left**, and may repeat on the right | One restaurant, joined to its several cuisine tags |
| `"many_to_one"` | The key may repeat on the **left**, and is unique on the **right** | Many inspections, joined to one record per restaurant |
| `"many_to_many"` | The key may repeat in both tables | Almost never what you want. This is the fan-out |

Our merge is `many_to_one`: one restaurant can be inspected many times, and each
restaurant should have exactly one row in the other table. Asking for `one_to_one`
instead is a stronger promise, and it is false here, because the same `restaurant_id`
appears on many different inspections. Run it and read the error message:


In [ ]:
try:
    # one_to_one requires a unique key in BOTH tables. rest_one now has a unique
    # restaurant_id, but insp does not, because a restaurant has many inspections.
    insp.merge(rest_one, on="restaurant_id", how="left", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", e)

Read the message closely, because it names *which* table failed the check. The first
`validate=` check we ran said "right dataset", because the restaurant table repeated the
key. This one says "left dataset", because the inspections table repeats it.

This also answers a question that sounds strange the first time you hear it. Why would
you ever want your code to stop with an error? Because the alternative is worse. Without
the check, pandas merges the tables and returns a result. You have a wrong answer, and
nothing on the screen tells you so. An error stops you at the moment the problem
happens, and it names the table that broke your assumption. A wrong table that raised no
error can end up in a finished report before anyone notices.


## Task 1.4: Reproducibility rules that apply from now on (10 minutes, for P2)

From this week on, every project repo must contain two things, kept up to date:

- **`README.md`**: what the project is, what each file is, how to run it, in ~15 lines.
- **Provenance, in writing**: for every data file, where it came from (URL), when it
  was downloaded, what license or terms covered it, and what has been done to it since.
  This is the **Provenance Brief** required in P2, so the same document serves both.

---
## Checkpoint: submit before leaving class

1. For the naive join: how many rows did we expect, and how many did we get?
2. Which summary statistic barely changed after the fan-out, and why does that make a
   fan-out join hard to detect?
3. What does `validate="one_to_one"` check, and how is that different from what
   `validate="many_to_one"` checks? Name one situation where an error is *beneficial* for your analysis.
4. P3 (projeect milestone 3) requires joining two project sources. Think about the two (or more) project sources that you have already identified. What is the grain of each, and what
   `validate=` argument will you use in that join?

*Answers here.*

---
## No Part 2 this week

**P2 is due Thursday, Oct 1, 11:59 PM**: audit, data dictionary, cleaning log, and Provenance Brief, per the P2 handout.

Submit this notebook to **Week 6 In-Class Activity** before leaving class: **Runtime >
Restart runtime**, **Run all**, download, rename to
`DSA405_002_FA26_Lab6_[yourUnityID].ipynb`.